# Auditing analytical SQL on real data

**Question:** What is the mean body mass among recorded measurements in the Palmer penguins snapshot, and which plausible analyses misstate that quantity?

This notebook is an independent methods demonstration with five constructed flawed-analysis cases. It is not a model transcript, employer assessment, or causal study. Source data: Horst, Hill & Gorman (2020), palmerpenguins, CC0-1.0; original study Gorman et al. (2014), doi:10.1371/journal.pone.0090081.

In [ ]:
import json
import html
import math
from IPython.display import HTML, display
from sql_analysis_audit.audit import (DATA, SQL, verify_snapshot, provenance, load_rows,
    create_database, query, build_report)

def table(rows):
    keys=list(rows[0]) if rows else []
    header=''.join('<th>'+html.escape(str(k))+'</th>' for k in keys)
    body=''.join('<tr>'+''.join('<td>'+html.escape(str(row[k]))+'</td>' for k in keys)+'</tr>' for row in rows)
    display(HTML('<table><thead><tr>'+header+'</tr></thead><tbody>'+body+'</tbody></table>'))

print('Verified SHA-256:', verify_snapshot())
print('Upstream commit:', provenance()['upstream_commit'])
connection=create_database(load_rows())
report=build_report()

## 1. Reconcile the sample before interpreting a mean

`COUNT(*)` counts source rows, while `COUNT(body_mass_g)` excludes missing masses. All `NA` fields remain SQL NULL. File-ordinal IDs are technical row keys, not biological subject identifiers.

In [ ]:
print((SQL/'profile.sql').read_text())
table(query(connection,'profile'))
assert report['profile']['n']==344
assert report['profile']['mass_observed']==342
assert report['profile']['complete_cases']==333

## 2. Species aggregation and window functions

The window denominator is the total number of observations; `DENSE_RANK` ranks the descriptive group means. These ranks do not measure a treatment effect or infer anything about unobserved populations.

In [ ]:
print((SQL/'species.sql').read_text())
table(query(connection,'species'))
assert math.isclose(sum(r['observation_share'] for r in report['species']),1.0)

## 3. Yearly sample changes with LAG

The difference from the preceding year's sample mean is not within-animal growth. The simplified table does not retain identities sufficient for that conclusion.

In [ ]:
print((SQL/'yearly.sql').read_text())
table(query(connection,'yearly'))
assert len(report['yearly'])==9

## 4. Averages of averages need an explicit estimand

Equal-species weighting is legitimate for a stated equal-species estimand. It does not recover the pooled observation mean. Weighting must use nonmissing mass counts, not total subgroup sizes.

In [ ]:
print((SQL/'weighted_mean.sql').read_text())
table(query(connection,'weighted_mean'))
assert math.isclose(report['weighted_means']['correct_pooled_mean_g'],report['profile']['mean_mass_g'])

## 5. Reproduce four computational review failures

The duplicate-tag relation is deliberately synthetic and used only to reproduce a many-to-many join mistake. It is not presented as real ecological metadata. Every candidate below is an authored negative control.

In [ ]:
for case in report['review_cases'][:4]:
    print(json.dumps(case,indent=2))
assert report['review_cases'][2]['candidate_rows']==496
assert report['review_cases'][2]['correct_rows']==344

## 6. Descriptive association is not causation

The species-centered calculation subtracts each species mean from both variables and correlates the residuals. It is not the average of species-specific correlations. Neither that calculation nor the pooled correlation controls every relevant sampling or causal mechanism.

In [ ]:
print((SQL/'association.sql').read_text())
table(report['association'])
print(json.dumps(report['review_cases'][4],indent=2))

## 7. Audit conclusion

Use the 342 recorded masses for the stated observed-mass mean. Preserve the missingness report; disclose any alternative sample or weighting choice. Reject the deliberately duplicated species dimension before joining. Report the pooled and species-centered associations only as descriptive quantities.

We do not attach p-values, confidence intervals, causal claims, predictive scores, or population claims to this small historical sample. SQL results are separately checked against independent Python calculations in the test suite.

In [ ]:
assert report['join_audit']['fact_rows']==report['join_audit']['joined_rows']==344
assert len(report['review_cases'])==5
assert build_report()==report
print('Runtime:', json.dumps(report['runtime'],sort_keys=True))
print('SQL fingerprints:', json.dumps(report['sql_sha256'],indent=2,sort_keys=True))
print('All notebook assertions passed.')
connection.close()